In [1]:
from pathlib import Path
from dotenv import load_dotenv

for _dir in [Path.cwd(), *Path.cwd().parents]:
    _env = _dir / '.env'
    if _env.is_file():
        load_dotenv(_env)
        break

In [2]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
# spark.sql('CREATE SCHEMA IF NOT EXISTS ddc_databricks.bronze')

DataFrame[]

In [3]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = 'ddc_databricks'
SCHEMA  = 'bronze'

BRONZE_TABLES = [
    'github_repos',
    'github_contributors',
    'github_readmes',
    'github_events',
    'pypi_metadata',
    'pypi_downloads_overall',
    'pypi_downloads_recent',
    'so_questions',
    'so_answers',
    'so_tag_info',
]

run_ts = datetime.utcnow().isoformat()
print(f'Tracking schema evolution for {len(BRONZE_TABLES)} bronze tables  (run: {run_ts})')

Tracking schema evolution for 10 bronze tables  (run: 2026-03-27T17:35:20.994756)


C:\Users\Behroz Karim\AppData\Local\Temp\ipykernel_145656\2832294133.py:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_ts = datetime.utcnow().isoformat()


In [ ]:
# Captures a schema snapshot (column names, types, nullability) for every Bronze Delta table.  
snapshot_rows = []

for tbl_name in BRONZE_TABLES:
    fqn = f'{CATALOG}.{SCHEMA}.{tbl_name}'
    try:
        df = spark.table(fqn)
    except Exception as exc:
        print(f'  SKIP {fqn}: {exc}')
        continue

    columns_meta = [
        {'name': f.name, 'type': f.dataType.simpleString(), 'nullable': f.nullable}
        for f in df.schema.fields
    ]

    snapshot_rows.append((
        tbl_name,
        len(df.schema.fields),
        json.dumps(columns_meta),
        df.schema.json(),
        run_ts,
    ))
    print(f'  {tbl_name}: {len(df.schema.fields)} columns')

current_snapshots_df = spark.createDataFrame(
    snapshot_rows,
    ['table_name', 'column_count', 'columns_json', 'schema_json', 'snapshot_at'],
)

print(f'\nSnapshots captured for {current_snapshots_df.count()} tables')
current_snapshots_df.select('table_name', 'column_count', 'snapshot_at').show(truncate=False)

  github_repos: 5 columns
  github_contributors: 5 columns
  github_readmes: 6 columns
  github_events: 5 columns
  pypi_metadata: 5 columns
  pypi_downloads_overall: 5 columns
  pypi_downloads_recent: 5 columns
  so_questions: 7 columns
  so_answers: 8 columns
  so_tag_info: 6 columns

Snapshots captured for 10 tables
+----------------------+------------+--------------------------+
|table_name            |column_count|snapshot_at               |
+----------------------+------------+--------------------------+
|github_repos          |5           |2026-03-27T17:35:20.994756|
|github_contributors   |5           |2026-03-27T17:35:20.994756|
|github_readmes        |6           |2026-03-27T17:35:20.994756|
|github_events         |5           |2026-03-27T17:35:20.994756|
|pypi_metadata         |5           |2026-03-27T17:35:20.994756|
|pypi_downloads_overall|5           |2026-03-27T17:35:20.994756|
|pypi_downloads_recent |5           |2026-03-27T17:35:20.994756|
|so_questions          |7    

In [ ]:
# Compares the current snapshot against the most recent previous one (if any).
schema_changes_rows = []

try:
    prev_snapshots = spark.table(f'{CATALOG}.{SCHEMA}.schema_snapshots')
    has_previous = prev_snapshots.count() > 0
except Exception:
    has_previous = False
    prev_snapshots = None

if has_previous:
    w = Window.partitionBy('table_name').orderBy(F.col('snapshot_at').desc())
    latest_prev = (
        prev_snapshots
        .withColumn('_rn', F.row_number().over(w))
        .filter(F.col('_rn') == 1)
        .drop('_rn')
    )
    prev_table_names = [r['table_name'] for r in latest_prev.select('table_name').collect()]

    for row in latest_prev.collect():
        tbl_name  = row['table_name']
        prev_cols = {c['name']: c for c in json.loads(row['columns_json'])}

        current_match = [r for r in snapshot_rows if r[0] == tbl_name]
        if not current_match:
            schema_changes_rows.append((tbl_name, 'TABLE_DROPPED', None, None, None, run_ts))
            continue

        curr_cols = {c['name']: c for c in json.loads(current_match[0][2])}

        for cname in curr_cols:
            if cname not in prev_cols:
                schema_changes_rows.append((
                    tbl_name, 'COLUMN_ADDED', cname,
                    None, curr_cols[cname]['type'], run_ts,
                ))

        for cname in prev_cols:
            if cname not in curr_cols:
                schema_changes_rows.append((
                    tbl_name, 'COLUMN_REMOVED', cname,
                    prev_cols[cname]['type'], None, run_ts,
                ))

        for cname in set(prev_cols) & set(curr_cols):
            if prev_cols[cname]['type'] != curr_cols[cname]['type']:
                schema_changes_rows.append((
                    tbl_name, 'TYPE_CHANGED', cname,
                    prev_cols[cname]['type'], curr_cols[cname]['type'], run_ts,
                ))
            if prev_cols[cname]['nullable'] != curr_cols[cname]['nullable']:
                old_null = 'nullable' if prev_cols[cname]['nullable'] else 'not_null'
                new_null = 'nullable' if curr_cols[cname]['nullable'] else 'not_null'
                schema_changes_rows.append((
                    tbl_name, 'NULLABLE_CHANGED', cname,
                    old_null, new_null, run_ts,
                ))

    for tbl_name, _, _, _, _ in snapshot_rows:
        if tbl_name not in prev_table_names:
            schema_changes_rows.append((tbl_name, 'TABLE_ADDED', None, None, None, run_ts))

if schema_changes_rows:
    schema_changes_df = spark.createDataFrame(
        schema_changes_rows,
        ['table_name', 'change_type', 'column_name', 'old_value', 'new_value', 'detected_at'],
    )
    print(f'Detected {len(schema_changes_rows)} schema change(s):')
    schema_changes_df.show(50, truncate=False)
else:
    if has_previous:
        print('No schema changes detected since the last snapshot.')
    else:
        print('First run — no previous snapshots to compare against. Baseline will be saved.')

First run — no previous snapshots to compare against. Baseline will be saved.


In [ ]:
# Logs detected changes: COLUMN_ADDED, COLUMN_REMOVED, TYPE_CHANGED, NULLABLE_CHANGED, TABLE_ADDED, TABLE_DROPPED
# Persists results to bronze.schema_snapshots and bronze.schema_changes (append mode for full history)
(
    current_snapshots_df
    .write
    .format('delta')
    .mode('append')
    .saveAsTable(f'{CATALOG}.{SCHEMA}.schema_snapshots')
)
print('bronze.schema_snapshots written (append mode)')

if schema_changes_rows:
    (
        schema_changes_df
        .write
        .format('delta')
        .mode('append')
        .saveAsTable(f'{CATALOG}.{SCHEMA}.schema_changes')
    )
    print('bronze.schema_changes written (append mode)')
else:
    print('bronze.schema_changes — nothing to write this run')

bronze.schema_snapshots written (append mode)
bronze.schema_changes — nothing to write this run


In [8]:
print('BRONZE SCHEMA EVOLUTION — RUN SUMMARY')
print(f'Run timestamp       : {run_ts}')
print(f'Tables tracked      : {len(snapshot_rows)}')
print(f'Total columns (all) : {sum(r[1] for r in snapshot_rows)}')
print(f'Schema changes found: {len(schema_changes_rows)}')
print()

if schema_changes_rows:
    print(' Changes detected ')
    schema_changes_df.show(truncate=False)
else:
    print('No changes.' if has_previous else 'First run — baseline snapshot saved.')

print()
print(' Column counts per table ')
current_snapshots_df.select('table_name', 'column_count').orderBy('table_name').show(truncate=False)

print('Tables written:')
print('  bronze.schema_snapshots  (append)')
print('  bronze.schema_changes    (append, if changes found)')

BRONZE SCHEMA EVOLUTION — RUN SUMMARY
Run timestamp       : 2026-03-27T17:35:20.994756
Tables tracked      : 10
Total columns (all) : 57
Schema changes found: 0

First run — baseline snapshot saved.

 Column counts per table 
+----------------------+------------+
|table_name            |column_count|
+----------------------+------------+
|github_contributors   |5           |
|github_events         |5           |
|github_readmes        |6           |
|github_repos          |5           |
|pypi_downloads_overall|5           |
|pypi_downloads_recent |5           |
|pypi_metadata         |5           |
|so_answers            |8           |
|so_questions          |7           |
|so_tag_info           |6           |
+----------------------+------------+

Tables written:
  bronze.schema_snapshots  (append)
  bronze.schema_changes    (append, if changes found)
